# **Data Auditing**

## Purpose
This notebook performs a structured **data audit** on the raw consumer complaint dataset before any cleaning or exploratory analysis.

The audit checks:

- dataset size and sample records
- schema and data types
- missing values
- duplicate rows and duplicate complaint IDs
- categorical value consistency
- product and sub-product relationships
- date validity and date sequence
- future dates
- blank values in text fields
- ZIP code format
- duplicate complaint narratives

# **1. Environment Setup and Dataset Discovery**

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
import os

drive_root = "/content/drive/MyDrive"

print("Searching for complaints_100k.parquet...\n")

found_files = []

for root, dirs, files in os.walk(drive_root):
    for file in files:
        if file.lower() == "complaints_100k.parquet":
            full_path = os.path.join(root, file)
            found_files.append(full_path)
            print("FOUND:")
            print(full_path)

if not found_files:
    print("File Not Found!")

Searching for complaints_100k.parquet...

FOUND:
/content/drive/MyDrive/Consumer Complain Project/data/raw/complaints_100k.parquet


# **2. Load the Raw Dataset**

In [ ]:
file_path = found_files[0]

df = pd.read_parquet(file_path)

print("Dataset loaded successfully!")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

Dataset loaded successfully!
Rows    : 90,112
Columns : 16


### 2.1 Preview the Data

In [ ]:
display(df.head())

,date_received,product,sub_product,issue,sub_issue,consumer_complaint_narrative,company_public_response,company,state,zip_code,tags,submitted_via,date_sent_to_company,company_response_to_consumer,timely_response,complaint_id
0,2026-04-25 20:09:10+00:00,Credit reporting or other personal consumer re...,Credit reporting,Problem with a company's investigation into an...,Investigation took more than 30 days,None,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",WA,98087,None,Web,2026-04-25 20:17:54+00:00,Closed with explanation,Yes,21603527
1,2026-02-14 21:18:34+00:00,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,None,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",AL,35022,None,Web,2026-02-14 21:19:00+00:00,Closed with explanation,Yes,19508852
2,2026-05-25 18:41:22+00:00,Credit reporting or other personal consumer re...,Credit reporting,Problem with a company's investigation into an...,Their investigation did not fix an error on yo...,None,None,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",TX,75287,None,Web,2026-05-25 18:45:42+00:00,None,Yes,22543000
3,2026-05-12 15:06:42+00:00,Debt collection,I do not know,Communication tactics,"You told them to stop contacting you, but they...",None,None,Western Management Consultants,NY,146XX,None,Web,2026-05-12 15:12:25+00:00,Closed with explanation,Yes,22116348
4,2026-05-01 08:25:14+00:00,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,None,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,SC,29455,None,Web,2026-05-01 08:25:37+00:00,Closed with explanation,Yes,21783248


### 2.2 Dataset Structure and Data Types

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90112 entries, 0 to 90111
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype              
---  ------                        --------------  -----              
 0   date_received                 90112 non-null  datetime64[us, UTC]
 1   product                       90112 non-null  object             
 2   sub_product                   90112 non-null  object             
 3   issue                         90112 non-null  object             
 4   sub_issue                     86095 non-null  object             
 5   consumer_complaint_narrative  20112 non-null  object             
 6   company_public_response       35420 non-null  object             
 7   company                       90112 non-null  object             
 8   state                         89882 non-null  object             
 9   zip_code                      89942 non-null  string             
 10  tags                          4714

In [ ]:
print("Data Types")
print("----------")

print(df.dtypes)

Data Types
----------
date_received                   datetime64[us, UTC]
product                                      object
sub_product                                  object
issue                                        object
sub_issue                                    object
consumer_complaint_narrative                 object
company_public_response                      object
company                                      object
state                                        object
zip_code                             string[python]
tags                                         object
submitted_via                                object
date_sent_to_company            datetime64[us, UTC]
company_response_to_consumer                 object
timely_response                              object
complaint_id                         string[python]
dtype: object


# **3. Dataset Overview and Basic Integrity**

In [ ]:
print("Dataset Overview")
print("----------------")
print(f"Total rows       : {df.shape[0]:,}")
print(f"Total columns    : {df.shape[1]:,}")
print(f"Duplicate rows   : {df.duplicated().sum():,}")
print(f"Duplicate complaint IDs : {df['complaint_id'].duplicated().sum():,}")

Dataset Overview
----------------
Total rows       : 90,112
Total columns    : 16
Duplicate rows   : 0
Duplicate complaint IDs : 0


In [ ]:
print("Unique Values in Each Column")
print("----------------------------")

for column in df.columns:
    print(column, ":", df[column].nunique())

Unique Values in Each Column
----------------------------
date_received : 88993
product : 16
sub_product : 58
issue : 92
sub_issue : 206
consumer_complaint_narrative : 18358
company_public_response : 11
company : 1499
state : 59
zip_code : 10829
tags : 3
submitted_via : 4
date_sent_to_company : 85806
company_response_to_consumer : 5
timely_response : 2
complaint_id : 90112


# **4. Missing Value Audit**

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

print("Missing Value Summary")
display(missing_summary)

Missing Value Summary


,missing_count,missing_percentage
tags,85398,94.77
consumer_complaint_narrative,70000,77.68
company_public_response,54692,60.69
company_response_to_consumer,15807,17.54
sub_issue,4017,4.46
date_sent_to_company,3180,3.53
state,230,0.26
zip_code,170,0.19
company,0,0.00
product,0,0.00


# **5. Categorical Value and Relationship Audit**
Here we check the consistency and distribution of important categorical fields. These checks help identify legacy categories, unexpected values, sparse groups, and product/sub-product relationships before cleaning.

### 5.1 Product Categories

In [ ]:
print("Product Categories")
print("------------------")
print(df["product"].value_counts(dropna=False))

Product Categories
------------------
product
Credit reporting or other personal consumer reports                             63088
Debt collection                                                                  9700
Checking or savings account                                                      4263
Credit card                                                                      4219
Money transfer, virtual currency, or money service                               2034
Mortgage                                                                         1758
Vehicle loan or lease                                                            1444
Payday loan, title loan, personal loan, or advance loan                          1121
Student loan                                                                     1011
Prepaid card                                                                      727
Debt or credit management                                                         655
Other   

In [ ]:
print("Unique Product Categories")
print("--------------------------")
for product in sorted(df["product"].unique()):
    print(product)

Unique Product Categories
--------------------------
Bank account or service
Checking or savings account
Consumer Loan
Credit card
Credit card or prepaid card
Credit reporting or other personal consumer reports
Credit reporting, credit repair services, or other personal consumer reports
Debt collection
Debt or credit management
Money transfer, virtual currency, or money service
Mortgage
Other
Payday loan, title loan, personal loan, or advance loan
Prepaid card
Student loan
Vehicle loan or lease


In [ ]:
print("Product Categories with Very Low Counts")
print("---------------------------------------")
print(df["product"].value_counts(dropna=False).tail(10))

Product Categories with Very Low Counts
---------------------------------------
product
Vehicle loan or lease                                                           1444
Payday loan, title loan, personal loan, or advance loan                         1121
Student loan                                                                    1011
Prepaid card                                                                     727
Debt or credit management                                                        655
Other                                                                             45
Credit reporting, credit repair services, or other personal consumer reports      43
Credit card or prepaid card                                                        2
Consumer Loan                                                                      1
Bank account or service                                                            1
Name: count, dtype: int64


In [ ]:
print("Low-Count Product Categories by Year")
print("-----------------------------------")

years = df["date_received"].dt.year

print(
    df[df["product"].isin([
        "Credit reporting, credit repair services, or other personal consumer reports",
        "Credit card or prepaid card",
        "Consumer Loan",
        "Bank account or service"
    ])]
    .groupby([years, "product"])
    .size()
)

Low-Count Product Categories by Year
-----------------------------------
date_received  product                                                                     
2014           Bank account or service                                                          1
2016           Consumer Loan                                                                    1
2019           Credit card or prepaid card                                                      2
2020           Credit reporting, credit repair services, or other personal consumer reports     3
2022           Credit reporting, credit repair services, or other personal consumer reports     5
2023           Credit reporting, credit repair services, or other personal consumer reports    35
dtype: int64


### 5.2 Product and Sub-Product Consistency

In [ ]:
print("Product and Sub-Product Combinations")
print("-----------------------------------")

product_subproduct = (
    df.groupby(["product", "sub_product"])
    .size()
    .reset_index(name="complaint_count")
    .sort_values("complaint_count", ascending=False)
)

print(product_subproduct.head(20))

Product and Sub-Product Combinations
-----------------------------------
                                              product  \
10  Credit reporting or other personal consumer re...   
18                                    Debt collection   
6                                         Credit card   
2                         Checking or savings account   
16                                    Debt collection   
21                                    Debt collection   
63                              Vehicle loan or lease   
34  Money transfer, virtual currency, or money ser...   
37                                           Mortgage   
60                                       Student loan   
48  Payday loan, title loan, personal loan, or adv...   
24                                    Debt collection   
7                                         Credit card   
25                                    Debt collection   
31  Money transfer, virtual currency, or money ser...   
38             

In [ ]:
print("Number of Sub-Products per Product")
print("--------------------------------")

print(
    df.groupby("product")["sub_product"]
    .nunique(dropna=True)
    .sort_values(ascending=False)
)

Number of Sub-Products per Product
--------------------------------
product
Debt collection                                                                 12
Mortgage                                                                         9
Payday loan, title loan, personal loan, or advance loan                          8
Money transfer, virtual currency, or money service                               7
Prepaid card                                                                     5
Checking or savings account                                                      4
Debt or credit management                                                        4
Credit card or prepaid card                                                      2
Credit card                                                                      2
Student loan                                                                     2
Credit reporting, credit repair services, or other personal consumer reports     2
Credit repo

In [ ]:
print("Sub-Products for Each Product")
print("----------------------------")

for product in df["product"].unique():
    print("\nProduct:", product)
    print(df[df["product"] == product]["sub_product"].value_counts())

Sub-Products for Each Product
----------------------------

Product: Credit reporting or other personal consumer reports
sub_product
Credit reporting                  62878
Other personal consumer report      210
Name: count, dtype: int64

Product: Debt collection
sub_product
I do not know                4993
Credit card debt             1436
Other debt                   1329
Rental debt                   580
Telecommunications debt       522
Medical debt                  329
Auto debt                     256
Payday loan debt              161
Private student loan debt      44
Federal student loan debt      25
Mortgage debt                  24
Credit card                     1
Name: count, dtype: int64

Product: Credit card
sub_product
General-purpose credit card or charge card    3645
Store credit card                              574
Name: count, dtype: int64

Product: Money transfer, virtual currency, or money service
sub_product
Mobile or digital wallet                            10

### 5.3 Other Important Categorical Fields

In [ ]:
print("Complaint Submission Channels")
print("-----------------------------")

print(df["submitted_via"].value_counts())

Complaint Submission Channels
-----------------------------
submitted_via
Web            89427
Phone            334
Postal mail      176
Referral         175
Name: count, dtype: int64


In [ ]:
print("Timely Response")
print("---------------")

print(df["timely_response"].value_counts())

Timely Response
---------------
timely_response
Yes    89009
No      1103
Name: count, dtype: int64


In [ ]:
print("Timely Response Validation")
print("--------------------------")

print(df["timely_response"].value_counts(dropna=False))

Timely Response Validation
--------------------------
timely_response
Yes    89009
No      1103
Name: count, dtype: int64


In [ ]:
print("Company Response to Consumer")
print("----------------------------")

print(df["company_response_to_consumer"].value_counts(dropna=False))

Company Response to Consumer
----------------------------
company_response_to_consumer
Closed with explanation            50849
None                               15807
Closed with non-monetary relief    11897
In progress                         9467
Closed with monetary relief         1576
Untimely response                    516
Name: count, dtype: int64


In [ ]:
print("Company Public Response Values")
print("------------------------------")
print(df["company_public_response"].value_counts(dropna=False))

Company Public Response Values
------------------------------
company_public_response
None                                                                                                                       54692
Company has responded to the consumer and the CFPB and chooses not to provide a public response                            34009
Company believes it acted appropriately as authorized by contract or law                                                    1131
Company believes the complaint provided an opportunity to answer consumer's questions                                         80
Company disputes the facts presented in the complaint                                                                         75
Company can't verify or dispute the facts in the complaint                                                                    34
Company believes complaint caused principally by actions of third party outside the control or direction of the company       33
Company bel

In [ ]:
print("Tags")
print("----")

print(df["tags"].value_counts(dropna=False))

Tags
----
tags
None                             85398
Servicemember                     2830
Older American                    1522
Older American, Servicemember      362
Name: count, dtype: int64


### 5.4 Category-Level Audit Summaries

In [ ]:
print("Top Complaint Issues")
print("--------------------")

print(df["issue"].value_counts(dropna=False).head(15))

Top Complaint Issues
--------------------
issue
Incorrect information on your report                               37917
Improper use of your report                                        13369
Problem with a company's investigation into an existing problem    12256
Attempts to collect debt not owed                                   4734
Managing an account                                                 2340
Written notification about debt                                     1637
Took or threatened to take negative or legal action                 1509
Problem with a purchase shown on your statement                     1492
False statements or representation                                  1131
Trouble during payment process                                       877
Problem with a lender or other company charging your account         697
Fraud or scam                                                        661
Dealing with your lender or servicer                                 598
Clo

In [ ]:
print("Top Complaint Sub-Issues")
print("-----------------------")

print(df["sub_issue"].value_counts(dropna=False).head(15))

Top Complaint Sub-Issues
-----------------------
sub_issue
Information belongs to someone else                                                 23338
Reporting company used your report improperly                                       11377
Account information incorrect                                                        8281
Investigation took more than 30 days                                                 6079
Their investigation did not fix an error on your report                              5552
None                                                                                 4017
Account status incorrect                                                             3619
Debt is not yours                                                                    2786
Credit inquiries on your report that you don't recognize                             1948
Personal information incorrect                                                       1542
Debt was result of identity theft        

In [ ]:
print("Top Companies by Complaint Count")
print("--------------------------------")

print(df["company"].value_counts(dropna=False).head(15))

Top Companies by Complaint Count
--------------------------------
company
Experian Information Solutions Inc.       21764
TRANSUNION INTERMEDIATE HOLDINGS, INC.    19214
EQUIFAX, INC.                             19050
Pending Company Match                      3207
BANK OF AMERICA, NATIONAL ASSOCIATION      1139
CAPITAL ONE FINANCIAL CORPORATION          1032
WELLS FARGO & COMPANY                      1005
CITIBANK, N.A.                              966
JPMORGAN CHASE & CO.                        794
Chime Financial Inc                         754
SYNCHRONY FINANCIAL                         730
Block, Inc.                                 713
Resurgent Capital Services L.P.             621
Paypal Holdings, Inc                        572
CL Holdings LLC                             546
Name: count, dtype: int64


In [ ]:
print("Top States by Complaint Count")
print("-----------------------------")

print(df["state"].value_counts(dropna=False).head(15))

Top States by Complaint Count
-----------------------------
state
TX    14161
FL    12160
CA     8734
GA     6212
NY     4534
IL     3658
NC     3160
PA     2935
NJ     2574
AL     2390
SC     2267
MS     2171
LA     2021
MD     2014
OH     1854
Name: count, dtype: int64


In [ ]:
print("Unique State Values")
print("--------------------")
print(sorted(df["state"].dropna().unique()))

Unique State Values
--------------------
['AA', 'AE', 'AK', 'AL', 'AP', 'AR', 'AS', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA', 'GU', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'PR', 'RI', 'SC', 'SD', 'TN', 'TX', 'UNITED STATES MINOR OUTLYING ISLANDS', 'UT', 'VA', 'VI', 'VT', 'WA', 'WI', 'WV', 'WY']


# **6. Date and Temporal Integrity Audit**

In [ ]:
print("Complaint Date Range")
print("--------------------")
print("First complaint:", df["date_received"].min())
print("Last complaint :", df["date_received"].max())

Complaint Date Range
--------------------
First complaint: 2014-07-24 00:33:39+00:00
Last complaint : 2026-07-06 09:05:16+00:00


In [ ]:
print("Future Date Check")
print("-----------------")

current_time = pd.Timestamp.now(tz="UTC")
future_dates = df[df["date_received"] > current_time]

print("Future complaint dates:", len(future_dates))

Future Date Check
-----------------
Future complaint dates: 0


In [ ]:
print("Date Sequence Check")
print("-------------------")

wrong_dates = df[
    df["date_sent_to_company"].notna()
    & (df["date_sent_to_company"] < df["date_received"])
]

print("Records where sent date is before received date:", len(wrong_dates))

Date Sequence Check
-------------------
Records where sent date is before received date: 0


# **7. Text and Identifier Integrity Audit**

In [ ]:
print("Duplicate Narratives (non-null)")
print("--------------------------------")

non_null_narratives = df["consumer_complaint_narrative"].dropna()
dup_narrative_count = non_null_narratives.duplicated().sum()

print("Total non-null narratives:", len(non_null_narratives))
print("Duplicate narratives:", dup_narrative_count)

Duplicate Narratives (non-null)
--------------------------------
Total non-null narratives: 20112
Duplicate narratives: 1754


In [ ]:
print("Blank Values in Text Columns")
print("----------------------------")

text_columns = [
    "product",
    "sub_product",
    "issue",
    "sub_issue",
    "consumer_complaint_narrative",
    "company_public_response",
    "company",
    "state",
    "tags",
    "submitted_via",
    "company_response_to_consumer",
    "timely_response"
]

blank_summary = {}

for column in text_columns:
    blank_summary[column] = (
        df[column].fillna("").astype(str).str.strip().eq("").sum()
    )

blank_summary = (
    pd.Series(blank_summary, name="blank_or_whitespace_count")
      .sort_values(ascending=False)
)

display(blank_summary.to_frame())

Blank Values in Text Columns
----------------------------


,blank_or_whitespace_count
tags,85398
consumer_complaint_narrative,70000
company_public_response,54692
company_response_to_consumer,15807
sub_issue,4017
state,230
product,0
sub_product,0
issue,0
company,0


# **8. ZIP Code Format Audit**

In [ ]:
print("ZIP Code Format Check")
print("---------------------")

zip_codes = df["zip_code"].dropna().astype(str)

# How many characters are in each ZIP code
zip_length_summary = zip_codes.str.len().value_counts().sort_index()
print("ZIP Length Distribution")
print(zip_length_summary)

# Some ZIP codes are masked by CFPB for privacy, e.g. "146XX" instead of "14601"
masked_count = 0
clean_count = 0
other_count = 0

for zip_value in zip_codes:
    if zip_value.isdigit() and len(zip_value) == 5:
        clean_count += 1
    elif "X" in zip_value.upper():
        masked_count += 1
    else:
        other_count += 1

print("\nClean 5-digit ZIP codes :", clean_count)
print("Masked ZIP codes (XX)   :", masked_count)
print("Other/unexpected values :", other_count)

ZIP Code Format Check
---------------------
ZIP Length Distribution
zip_code
5    89942
Name: count, dtype: int64

Clean 5-digit ZIP codes : 80854
Masked ZIP codes (XX)   : 9088
Other/unexpected values : 0


In [ ]:
print("ZIP Code Format Check")
print("---------------------")

zip_length_summary = (
    df["zip_code"]
      .dropna()
      .astype(str)
      .str.len()
      .value_counts()
      .sort_index()
)

print(zip_length_summary)

ZIP Code Format Check
---------------------
zip_code
5    89942
Name: count, dtype: int64


# **9. Audit Summary and Handoff to Data Cleaning**

## Key Findings

**Size and integrity**
- 90,112 rows, 16 columns. No duplicate rows, no duplicate complaint IDs.

**Missing values**
- `tags`: 94.77% missing, `consumer_complaint_narrative`: 77.68% missing,
  `company_public_response`: 60.69% missing — These columns have high missingness and require explicit treatment decisions during cleaning.
- `sub_issue`: 4.46% missing, `date_sent_to_company`: 3.53% missing,
  `state`: 0.26% missing, `zip_code`: 0.19% missing.

**Categorical consistency**
- `product` contains legacy/retired category names alongside current ones
  (e.g. "Consumer Loan", "Bank account or service", "Credit card or prepaid card"),
  each with very low counts (1–43 rows) — these need consolidation into current categories.
- `state` contains non-standard values: military/territory codes (AA, AE, AP, GU, PR, VI, AS)
  and one non-2-letter value, "UNITED STATES MINOR OUTLYING ISLANDS" — needs standardization.

**Dates**
- No future-dated complaints. No records where `date_sent_to_company` precedes `date_received`.
  Date fields pass integrity checks as-is.

**Text and identifiers**
- 1,754 duplicate complaint narratives out of 20,112 non-null — worth flagging in cleaning
  in case they represent boilerplate/templated complaints rather than true duplicates.

**ZIP codes**
- ZIP codes are a mix of clean 5-digit values and CFPB-masked values (e.g. "146XX").
  Masked ZIPs need a decision: keep as-is, treat as partial geography, or drop for
  ZIP-level analysis.

## Decisions for `02_Data_Cleaning.ipynb`
1. Standardize legacy `product` category names into their current equivalents.
2. Standardize/clean non-standard `state` values.
3. Decide treatment for masked ZIP codes.
4. Decide whether blank/whitespace values should be recoded as missing.
5. Decide whether duplicate narratives should be deduplicated or kept.

**Note:** This notebook is read-only — no changes were made to the raw dataset.
All cleaning decisions above are implemented in a separate working dataframe in
`02_Data_Cleaning.ipynb`.